# Step By Step 01 - 入门

Write and read Parquet, then calculate a first Arrow-native factor.


In [ ]:
import asyncio
import datetime as dt
import tempfile
import pyarrow as pa
import alphamind
from alphamind.data import Transformer, read_parquet, write_parquet
from alphamind.formula import CSRank

source = pa.table({
    "trade_date": [dt.date(2026, 1, 5), dt.date(2026, 1, 5), dt.date(2026, 1, 6), dt.date(2026, 1, 6)],
    "code": ["A", "B", "A", "B"],
    "value": [1.0, 2.0, 3.0, 4.0],
})
with tempfile.TemporaryDirectory() as directory:
    path = f"{directory}/input.parquet"
    write_parquet(source, path)
    loaded = read_parquet(path)
    result = await asyncio.to_thread(Transformer({"rank": CSRank("value")}).transform, "code", loaded)
assert result["rank"].to_pylist() == [1.0, 2.0, 1.0, 2.0]
{"version": alphamind.__version__, "rank": result["rank"].to_pylist()}
